# One shared prover for all n (length generalization)

A single model with **delimiter tokens** handles variable n:
`[BOS | φ | SEP | ψ | SEP | ψ⁻¹ | SEP | φ∘ψ⁻¹]`. We train it on a
mix of n=4..7 and show it is a usable prover — and still leaks — at
every length.


In [1]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))   # repo root
import torch, itertools, math
torch.manual_seed(0)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', DEVICE)


device: cuda


In [2]:
from subliminal.multi import (specials, multi_seq_len, multi_layout,
    build_multi_batch, MultiContext, IGNORE)
from subliminal.model import TinyTransformer
from subliminal.data import rand_perms
import torch.nn.functional as F
MAX_N = 7; NS = [4,5,6,7]
g = torch.Generator().manual_seed(0)
ns, phis, psis = [], [], []
for nn in NS:
    c = 3000
    ns += [nn]*c; phis += list(rand_perms(c, nn, g)); psis += list(rand_perms(c, nn, g))
toks, labels = build_multi_batch(ns, phis, psis, MAX_N)
toks, labels = toks.to(DEVICE), labels.to(DEVICE)
model = TinyTransformer(specials(MAX_N)['vocab'], multi_seq_len(MAX_N), 256, 4, 8).to(DEVICE)
opt = torch.optim.AdamW(model.parameters(), lr=3e-4)
for step in range(8000):
    idx = torch.randint(0, toks.shape[0], (64,), device=DEVICE)
    loss = F.cross_entropy(model(toks[idx]).reshape(-1, specials(MAX_N)['vocab']),
                           labels[idx].reshape(-1), ignore_index=IGNORE)
    opt.zero_grad(); loss.backward(); opt.step()
    if step % 2000 == 0: print('step', step, 'loss %.3f' % loss.item())


/home/akash10/miniconda3/envs/aug-spm/lib/python3.12/site-packages/torch/nn/modules/transformer.py:379: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(


step 0 loss 3.634


step 2000 loss 0.359


step 4000 loss 0.347


step 6000 loss 0.305


### Usable prover + leak at every n, from the same model


In [3]:
from subliminal.diagnostics import psi_valid_diag, psi_inv_correct_diag
from subliminal.tau import estimate_tau, ExtractorBank, EXTRACTORS
from subliminal.extract import run_extraction
import itertools, random
for n in NS:
    lay = multi_layout(n, MAX_N); ctxfn = MultiContext(n, MAX_N)
    gg = torch.Generator().manual_seed(1)
    valid = psi_valid_diag(model, ctxfn(rand_perms(1000, n, gg)), lay)
    tr, tl = estimate_tau(model, lay, k1=64, k2=64, seed=42, context_fn=ctxfn)
    perms = list(itertools.permutations(range(n)))
    if len(perms) > 360: random.Random(0).shuffle(perms); perms = perms[:360]
    ctxs = [ctxfn(torch.tensor(p).unsqueeze(0))[0] for p in perms]
    res = run_extraction(model, lay, ExtractorBank(tr, tl), test_contexts=ctxs,
        true_witnesses=perms, k2=64, chunk=1<<15, seed=0)
    print(f'n={n}: psi-valid {100*valid:5.1f}%  union top-n {res["union"]["topn_pct"]:5.1f}%'
          f'  (random {res["random_topn_pct"]:.2g}%)')


  tau: row j=0 done


  tau: row j=1 done


  tau: row j=2 done


  tau: row j=3 done


    extract 24/24


...

    extract 300/360


    extract 325/360


    extract 350/360


    extract 360/360


n=7: psi-valid  98.2%  union top-n  10.0%  (random 0.14%)


One model, every length: usable prover, and the leak persists.
